# AMR-UNBIND -- cloud run (Google Colab)

This notebook installs AMR-UNBIND and runs **the same pipeline the desktop app uses**, on a free Colab GPU, with this run's inputs already filled in below.

**Before running anything:** go to **Runtime -> Change runtime type** and select a **T4 GPU** (or better), then run the cells below in order (Runtime -> Run all).

The last cell downloads a zip of the results back to your browser when the run finishes.

In [ ]:
!nvidia-smi

In [ ]:
import pathlib, urllib.request, zipfile
urllib.request.urlretrieve("https://github.com/MyMKGitH/AMR-UNBIND/archive/refs/heads/main.zip", 'amr_unbind_src.zip')
with zipfile.ZipFile('amr_unbind_src.zip') as zf:
    zf.extractall('.')
project_dirs = [p for p in pathlib.Path('.').iterdir() if p.is_dir() and p.name.lower().startswith('amr-unbind')]
assert project_dirs, 'Could not find an extracted AMR-UNBIND-* directory.'
%cd {project_dirs[0]}

In [ ]:
# Installs the exact dependency set requirements-docking.txt pins
# for this version, so this notebook can't silently drift from what
# the desktop app actually runs.
!pip install -q -r requirements-docking.txt
!pip install -q -e .

## This run's inputs (already filled in)

In [ ]:
from pathlib import Path

PDB_PATH = None
PDB_ID = "3f2r"

In [ ]:
POSE_SDF_PATH = None

In [ ]:
SMILES = "CC[N+](C)(C)CCOP(=O)(O)O"
CORE_CHAIN = "A"
CORE_RESIDUE = 122
EXIT_CHAIN = "A"
EXIT_RESIDUE = 205
AUTOMATIC_DOCKING = True
DOCKING_BOX_ANGSTROM = 22.0
DOCKING_EXHAUSTIVENESS = 8
DOCKING_CENTER_ANGSTROM = None
CONFIG_DICT = {
  "temperature_kelvin": 300.0,
  "pressure_atm": 1.0,
  "friction_per_ps": 1.0,
  "timestep_fs": 2.0,
  "ionic_strength_molar": 0.15,
  "solvation_padding_nm": 1.0,
  "nvt_equilibration_ps": 10.0,
  "npt_equilibration_ps": 10.0,
  "pull_distance_nm": 0.25,
  "pull_velocity_nm_per_ps": 0.005,
  "steering_k_kj_mol_nm2": 500.0,
  "report_interval_steps": 100,
  "checkpoint_interval_steps": 5000,
  "minimization_iterations": 500,
  "seed": 20260907,
  "protein_restraint_k_kj_mol_nm2": 100.0,
  "pH": 7.4,
  "forcefield_protein": "amber14-all.xml",
  "forcefield_water": "amber14/tip3p.xml",
  "small_molecule_forcefield": "openff-2.2.1",
  "preferred_platform": "auto",
  "cuda_device_index": "0"
}

## Run AMR-UNBIND

This is the same `AMRUnbindPipeline` the desktop app calls.

In [ ]:
from amr_unbind.config import SimulationConfig
from amr_unbind.pipeline import AMRUnbindPipeline

import time as _time
_last_stage = {"name": None, "started": _time.time()}

def _progress(stage, fraction, detail=""):
    now = _time.time()
    if _last_stage["name"] != stage:
        _last_stage["name"] = stage
        _last_stage["started"] = now
    elapsed = now - _last_stage["started"]
    line = f'[{fraction:6.1%}] {stage} ({elapsed:.0f}s in this stage)'
    if detail:
        line += f' -- {detail}'
    print(line, flush=True)

config = SimulationConfig.from_dict(CONFIG_DICT)
pipeline = AMRUnbindPipeline(work_dir=Path('./amr_workspace'))
run_dir = pipeline.run(
    pdb_path=PDB_PATH,
    pdb_id=PDB_ID,
    smiles=SMILES,
    core_chain=CORE_CHAIN,
    core_residue=CORE_RESIDUE,
    exit_chain=EXIT_CHAIN,
    exit_residue=EXIT_RESIDUE,
    config=config,
    ligand_pose_sdf=POSE_SDF_PATH,
    automatic_docking=AUTOMATIC_DOCKING,
    docking_box_angstrom=DOCKING_BOX_ANGSTROM,
    docking_exhaustiveness=DOCKING_EXHAUSTIVENESS,
    docking_center_angstrom=tuple(DOCKING_CENTER_ANGSTROM) if DOCKING_CENTER_ANGSTROM else None,
    progress_callback=_progress,
)
print('Run complete:', run_dir)

## Download the results

In [ ]:
import shutil
from google.colab import files

archive_path = shutil.make_archive(run_dir.name, 'zip', run_dir)
files.download(archive_path)